In [27]:
import pandas as pd
import xgboost as xgb
import numpy as np
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score

In [28]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Reduced Dataset.csv')

In [29]:
# Drop diseases with less than 1000 instances
disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= 750].index
df = df[df['diseases'].isin(valid_diseases)]

# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("Number of remaining classes in training set:", len(np.unique(y_train)))
print("Number of rows left:", len(df))

Number of remaining classes in training set: 114
Number of rows left: 114312


In [31]:
# Apply SMOTE for class balancing in the training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Print the new class distribution after SMOTE
print("Class distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())
print("Number of remaining classes in training set:", len(np.unique(y_train_resampled)))
# Print the number of rows in the resampled training set
print("Number of rows in the resampled training set:", len(X_train_resampled))

Class distribution after SMOTE:
20     1002
9      1002
66     1002
80     1002
1      1002
       ... 
40     1002
106    1002
54     1002
102    1002
88     1002
Name: count, Length: 114, dtype: int64
Number of remaining classes in training set: 114
Number of rows in the resampled training set: 114228


In [38]:
def objective(trial):
    params = {
        'objective': 'multi:softprob',
        'num_class': len(np.unique(y_train)),
        'tree_method': 'hist',
        'eval_metric': 'mlogloss',
        
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        
        # Try including early stopping directly in the model parameters
        'early_stopping_rounds': 50,
    }
    
    # Create XGBoost model with parameters from Optuna
    model = xgb.XGBClassifier(**params)
    
    # In XGBoost 3.0.0, try a simpler fit call
    model.fit(
        X_train_resampled,
        y_train_resampled,
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    
    preds = model.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    return accuracy

In [40]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize',  # Assuming you are maximizing accuracy
    study_name="XGboost_diseases_symptoms_dropextremelymore750withSMOTE_study", 
    storage=r"sqlite:///C:/Users/khiew/Downloads/xgboost.db", 
    load_if_exists=True  # Load the study if it already exists, to resume from the last trial
)

# Optimize the study with your objective function, you can adjust the n_trials as needed
study.optimize(objective, n_trials=20)
# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

[I 2025-04-23 12:13:19,140] A new study created in RDB with name: XGboost_diseases_symptoms_dropextremelymore750withSMOTE_study
[I 2025-04-23 12:13:43,265] Trial 0 finished with value: 0.48134540523990726 and parameters: {'max_depth': 9, 'learning_rate': 0.29074431330795825, 'n_estimators': 613, 'subsample': 0.8696197024234646, 'colsample_bytree': 0.9624042947257408, 'gamma': 4.274017224314193, 'reg_alpha': 3.733896279144183, 'reg_lambda': 0.6389334231830462}. Best is trial 0 with value: 0.48134540523990726.
[I 2025-04-23 12:16:53,329] Trial 1 finished with value: 0.4843196430914578 and parameters: {'max_depth': 9, 'learning_rate': 0.25078793423192886, 'n_estimators': 902, 'subsample': 0.7240571647845493, 'colsample_bytree': 0.9294469437986312, 'gamma': 1.9930746269107391, 'reg_alpha': 3.0531682419929265, 'reg_lambda': 3.06368994828134}. Best is trial 1 with value: 0.4843196430914578.
[I 2025-04-23 12:19:04,394] Trial 2 finished with value: 0.48458207584306523 and parameters: {'max_dep


Best Trial:
FrozenTrial(number=12, state=TrialState.COMPLETE, values=[0.48794996282202685], datetime_start=datetime.datetime(2025, 4, 23, 12, 44, 19, 148398), datetime_complete=datetime.datetime(2025, 4, 23, 12, 49, 50, 509384), params={'max_depth': 6, 'learning_rate': 0.12753786953325266, 'n_estimators': 980, 'subsample': 0.796257479625381, 'colsample_bytree': 0.8140477189469144, 'gamma': 1.2101343529636543, 'reg_alpha': 0.2705240408855737, 'reg_lambda': 4.9576976971616125}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'max_depth': IntDistribution(high=15, log=False, low=3, step=1), 'learning_rate': FloatDistribution(high=0.3, log=False, low=0.01, step=None), 'n_estimators': IntDistribution(high=1000, log=False, low=100, step=1), 'subsample': FloatDistribution(high=1.0, log=False, low=0.5, step=None), 'colsample_bytree': FloatDistribution(high=1.0, log=False, low=0.5, step=None), 'gamma': FloatDistribution(high=5.0, log=False, low=0.0, step=None), 'reg_alpha